# TF32 — Hiệu chỉnh ViSoBERT vòng hai

Chọn Runtime → Change runtime type → T4 GPU. Upload `colab-input.zip` trong thư mục `tf32-visobert-round2` khi được yêu cầu. Bộ dev gồm 240 câu đã duyệt. Có 235 kết quả được ghép theo nguyên văn, 5 câu cần suy luận mới.

Gói đã chuẩn bị ngày 11/11/2026 có cache OpenAI đủ 240 câu. Nếu tự tái tạo gói input, chạy lệnh trong hướng dẫn để bổ sung cache trước bước upload. Mỗi tín hiệu Core được tính là gắn cờ nghi vấn trong baseline. Báo cáo đo phát hiện vi phạm, chưa đo hành vi tự động ẩn.

In [ ]:
%pip install -q "transformers==4.51.0" "tokenizers==0.21.2" sentencepiece safetensors \
    "pydantic>=2.6,<3" pyyaml

In [ ]:
import io
import json
import sys
import zipfile
from pathlib import Path

from google.colab import files

uploaded = files.upload()
assert len(uploaded) == 1, "Chọn đúng một file colab-input.zip"
bundle_bytes = next(iter(uploaded.values()))
bundle_path = Path("/content/tf32-round2-input.zip")
bundle_path.write_bytes(bundle_bytes)
sys.path.insert(0, str(bundle_path))
from calibrate_visobert import (  # noqa: E402 - module comes from uploaded bundle
    MODEL,
    REVISION,
    calibrate,
    evaluate_test,
    merge_openai,
    validate_rows,
)

from app.services.summary.pii import mask_pii  # noqa: E402 - bundled source

with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as archive:
    dev = json.loads(archive.read("dev-pack.json"))
print("Dev samples", len(dev["rows"]), "new predictions", dev.get("pending_ids", []))

## Bổ sung dự đoán ViSoBERT

Cell này tải đúng checkpoint đã dùng trong lần thử đầu, rồi chỉ chạy những câu còn thiếu. GPU T4 được ưu tiên khi có.

In [ ]:
import hashlib
import time

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = None
model = None


def infer_rows(rows):
    global tokenizer, model
    if model is None:
        tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION, trust_remote_code=False)
        model, info = AutoModelForSequenceClassification.from_pretrained(
            MODEL,
            revision=REVISION,
            trust_remote_code=False,
            use_safetensors=True,
            output_loading_info=True,
        )
        assert not info["missing_keys"] and not info["mismatched_keys"], info
        assert model.config.num_labels == 3
        model.to(device).eval()
    predictions = {}
    start = time.perf_counter()
    batch_size = 64 if device.type == "cuda" else 16
    with torch.inference_mode():
        for offset in range(0, len(rows), batch_size):
            batch = rows[offset : offset + batch_size]
            encoded = tokenizer(
                [mask_pii(row["text"]) for row in batch],
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt",
            ).to(device)
            probabilities = model(**encoded).logits.softmax(dim=-1)
            assert torch.isfinite(probabilities).all()
            for row, (clean, offensive, hate) in zip(batch, probabilities.tolist(), strict=True):
                predictions[row["id"]] = {
                    "clean": clean,
                    "offensive": offensive,
                    "hate": hate,
                    "toxic": offensive + hate,
                }
    print(len(rows), "rows", time.perf_counter() - start, "seconds", device)
    return predictions


pending = [r for r in dev["rows"] if "clean" not in dev["scores"][r["id"]]]
for key, prediction in infer_rows(pending).items():
    dev["scores"][key].update(prediction)

## Nhận cache OpenAI và khóa ngưỡng

Upload `openai-dev-cache.json` đã bổ sung đủ 240 câu. Budget FPR tăng tối đa 0,01 tương đương 1 điểm phần trăm là cấu hình thử nghiệm. Với 122 câu sạch, budget này cho phép thêm tối đa 1 false positive. Ngưỡng 1,01 trong bảng sweep dùng để tắt nhánh tương ứng.

Runner chọn recall harassment/hate cao nhất trong budget, ưu tiên ít false positive rồi F1 khi hòa điểm.

In [ ]:
import pandas as pd
from IPython.display import display

uploaded_cache = files.upload()
assert len(uploaded_cache) == 1
merge_openai(dev, json.loads(next(iter(uploaded_cache.values())).decode("utf-8-sig")))
dev.pop("pending_ids", None)
lock = calibrate(dev, max_fpr_delta=0.01)
display(
    pd.DataFrame(
        [
            {"method": "OpenAI + Core", **lock["dev_baseline"]},
            {"method": "OpenAI + Core + ViSoBERT", **lock["dev_metrics"]},
        ]
    )
)
print("Frozen config", lock["config"])
display(
    pd.DataFrame([{**entry["config"], **entry["metrics"]} for entry in lock["sweep"]]).sort_values(
        ["target_recall", "fp"], ascending=[False, True]
    )
)
output_zip = Path("/content/tf32-round2-dev-results.zip")
with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.writestr("dev-complete.json", json.dumps(dev, ensure_ascii=False, allow_nan=False))
    archive.writestr("frozen-config.json", json.dumps(lock, ensure_ascii=False, allow_nan=False))
files.download(str(output_zip))

## Khi đã có test mới

Chạy hai cell sau để tạo dự đoán ViSoBERT cho file `test.jsonl` đã được duyệt. Mỗi dòng cần `id`, `text`, `toxic` dạng boolean, `kind`, `group_id`. Những câu cùng gốc/template dùng chung group_id. Dữ liệu đưa lên Colab cần được khử thông tin cá nhân trước.

Sau khi tải `visobert-test.json`, dùng lệnh trong hướng dẫn trên Windows để lấy cache OpenAI, chạy luật Core thật và tạo `test-pack.json`. Upload test pack ở cell cuối. Cấu hình chấm test lấy từ `frozen-config.json` đã tải xuống.

In [ ]:
test_upload = files.upload()
assert len(test_upload) == 1
test_bytes = next(iter(test_upload.values()))
test_rows = [
    json.loads(line) for line in test_bytes.decode("utf-8-sig").splitlines() if line.strip()
]
validate_rows(test_rows)
test_predictions = infer_rows(test_rows)
test_result = {
    "model": MODEL,
    "revision": REVISION,
    "sha256": hashlib.sha256(test_bytes).hexdigest(),
    "predictions": test_predictions,
    "device": str(device),
    "max_length": 256,
    "torch_version": torch.__version__,
}
prediction_path = Path("/content/visobert-test.json")
prediction_path.write_text(json.dumps(test_result, allow_nan=False), encoding="utf-8")
files.download(str(prediction_path))

## Chấm test bằng cấu hình đã khóa

Upload ba file `dev-complete.json`, `frozen-config.json`, `test-pack.json` cùng lúc. Chạy cell này dùng cấu hình đã lưu, có kiểm tra hash dev, phiên bản model/rules/policy và trùng ID, câu gốc, nội dung lịch sử. Trùng nghĩa qua cách diễn đạt khác vẫn cần người duyệt kiểm tra.

In [ ]:
test_files = files.upload()
required = {"dev-complete.json", "frozen-config.json", "test-pack.json"}
assert required.issubset(test_files), "Cần đủ ba file đúng tên"
frozen_dev = json.loads(test_files["dev-complete.json"])
frozen_lock = json.loads(test_files["frozen-config.json"])
test_pack = json.loads(test_files["test-pack.json"])
report = evaluate_test(frozen_dev, test_pack, frozen_lock)
display(
    pd.DataFrame(
        [
            {"method": "OpenAI + Core", **report["baseline"]},
            {"method": "OpenAI + Core + ViSoBERT", **report["hybrid"]},
        ]
    )
)
print("Recall delta", report["target_recall_delta"], "FPR delta", report["fpr_delta"])
print("Meets proposed gate", report["meets_proposed_gate"])
report_path = Path("/content/tf32-round2-test-report.json")
report_path.write_text(json.dumps(report, ensure_ascii=False, allow_nan=False), encoding="utf-8")
files.download(str(report_path))